In [0]:
from pyspark.sql.functions import lower,trim,when,col,countDistinct
import datetime

#### 1.Create and get widgets

In [0]:
sourceSystem = dbutils.widgets.get("sourceSystem").lower()

#### 2.Calling params and utils  notebook

In [0]:
%run ../utils/params

In [0]:
%run ../utils/utilsMethods

####3.Calling logging notebook

In [0]:
%run ../utils/loggingNotebook

In [0]:
notebookPath = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
notebookName = notebookPath.split("/")[-1]
jobID = dbutils.notebook.entry_point.getDbutils().notebook().getContext().tags().get("jobId")
adbpipelineID=f"{sourceSystem}_{notebookName}_{jobID}"
properties={'custom_dimensions': {'notebookName':f'{notebookName}', 'jobID':f'{jobID}' ,'adbPipelineID':f'{adbpipelineID}'}}

In [0]:
logger.info("Reading the config information file from given location", extra = properties)
        
configDf = readSparkCsv(jobConfigPath)
configDf = configDf.select([when(trim(lower(col(c)))=="null",None).otherwise(col(c)).alias(c) for c in configDf.columns])

configDfFiltered=configDf.filter(trim(lower(configDf.sourceSystem))==(sourceSystem).lower())
configDf = configDfFiltered.withColumn("activeFlag", trim(lower(configDfFiltered["activeFlag"])))\
                               .withColumn("sourceSystem",trim(lower(configDfFiltered["sourceSystem"])))\
                               .withColumn("workloadType",trim(lower(configDfFiltered["workloadType"])))

####4.Last Run Updation

In [0]:
try:
    jobControlDf=spark.sql(f"select * from {jobControlTable} where sourceSystem='{sourceSystem}'")
    if jobControlDf.count() > 0:
        if (jobControlDf.filter((col("lastRunDate")=="null")) or  configDf.filter((col("workloadType")=='incremental') | (col("workloadType")=='histincr'))):
            updateQuery=f"""UPDATE {jobControlTable} SET lastRunDate = currentRunDate WHERE sourceSystem = '{sourceSystem}'"""
            spark.sql(updateQuery)
    
            logger.info("Updation of lastRunDate with value of currentRunDate column of jobControl table is done.",extra=properties)
    
except Exception as e:
    logger.exception("Updation of lastRunDate with value of currentRunDate column of jobControl table has failed",extra=properties)
    raise e

####5.Creating backup for userConfig files 

In [0]:
try:
    folderPaths = [jobConfigPath, ruleMasterPath, ruleObjectMappingPath]

    # Define the path of your destination folder
    lastRunDateValue= spark.sql(f"select * from {jobControlTable} where sourceSystem='{sourceSystem}'").select("lastRunDate").collect()[0][0].strftime('%Y-%m-%d')
    destinationFolderPath = f'{csvBackupPath}{lastRunDateValue}'

    # Create the destination folder if it does not exist
    dbutils.fs.mkdirs(destinationFolderPath)

    for folderPaths in folderPaths:
    
        for fileName in dbutils.fs.ls(folderPaths):
            # Get the file path
            filePath = fileName.path

            # Get the file name without the path
            fileNameWithoutPath = filePath.split("/")[-1]

            destinationFilePath = f'{destinationFolderPath}/{fileNameWithoutPath}'

            dbutils.fs.cp(filePath, destinationFilePath, True)
            
    logger.info("Archival of userConfig File is done for this '{lastRunDateValue}'",extra=properties)
except exception as e:
    logger.exception(f"Archival of userConfig File has failed: {str(e)}",extra=properties)
    raise e

In [0]:
dbutils.notebook.exit("Notebook ran successfully")